In [35]:
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    roc_auc_score
)
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
import xgboost as xgb
import time
from pathlib import Path

In [36]:
THRESHOLD = 0.25
RANDOM_STATE = 30
FOLDER_OUTPUT = Path('data') / 'data_models'

In [37]:
requirments_name = []

with open("requirments.txt", "r", encoding="utf-8") as requirments:
    for r in requirments:
        text = r.replace("\n", "")
        requirments_name.append(text.split("=="))

for requirment_name in requirments_name:
    print(f"{requirment_name[0]} == {requirment_name[1]}")

version_system = sys.version.split()
print(f"version Python == {(version_system[0])}")


numpy == 2.5.4
pandas == 3.1.1
jupyter == 1.1.1
matplotlib == 3.11.2
ucimlrepo == 0.0.7
scikit-learn == 1.9.1
xgboost == 3.4.1
version Python == 3.12.10


In [38]:
data = []

with open('data/adult.data', 'r', encoding='utf-8') as file:
    for line in file:
      clear_data = line.replace("\n", "").split(", ")
      data.append(clear_data)


In [39]:
data_test = []

with open("data/adult.test", "r", encoding="utf-8") as file:
    for line in file:
        clear_data = line.replace("\n", "").replace('.', '').split(", ")
        data_test.append(clear_data)


In [40]:
columns = []

with open("data/adult.names", "r", encoding="utf-8") as file:
    lines = file.readlines()
    for line in range(96, 110):
      clear_data = lines[line].replace('\n', '').split(':')
      columns.append(clear_data[0])


columns.append('income')


In [41]:
df = pd.DataFrame(data, columns=columns)
df_test = pd.DataFrame(data_test, columns=columns)

C:\Users\TgSBot_xD\AppData\Local\Temp\ipykernel_13128\971865605.py:1: Pandas4Warning: Constructing a DataFrame from a list of sequences with mismatched lengths is deprecated and will raise in a future version. The shorter sequences are currently padded with NaN; make all sequences the same length before constructing instead.
  df = pd.DataFrame(data, columns=columns)
C:\Users\TgSBot_xD\AppData\Local\Temp\ipykernel_13128\971865605.py:2: Pandas4Warning: Constructing a DataFrame from a list of sequences with mismatched lengths is deprecated and will raise in a future version. The shorter sequences are currently padded with NaN; make all sequences the same length before constructing instead.
  df_test = pd.DataFrame(data_test, columns=columns)


In [42]:
index_null_value = df.index[df.isna().any(axis=1)]
index_null_value_test = df_test.index[df_test.isna().any(axis=1)]

if not len(index_null_value) == 0:
    print(f"Delete line from train data frame - {index_null_value.to_list()}")
    df.iloc[index_null_value]
    df.drop(index_null_value.to_list(), inplace=True)

else:
  print("doesn't have empty line for train data frame")


if not len(index_null_value_test) == 0:
    print(f"Delete line from train data frame - {index_null_value_test.to_list()}")
    df_test.iloc[index_null_value_test]
    df_test.drop(index_null_value_test.to_list(), inplace=True)

else:
    print("doesn't have empty line")

Delete line from train data frame - [32561]
Delete line from train data frame - [16281]


C:\Users\TgSBot_xD\AppData\Local\Temp\ipykernel_13128\1686725349.py:7: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(index_null_value.to_list(), inplace=True)
C:\Users\TgSBot_xD\AppData\Local\Temp\ipykernel_13128\1686725349.py:16: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_test.drop(index_null_value_test.to_list(), inplace=True)


In [43]:
df_test = df_test.replace('?', 'unknown')
df = df.replace("?", "unknown")

In [44]:
object_columns = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
]

numerical_columns = list(set(columns) - set(object_columns))
numerical_columns.remove("fnlwgt")

target_index_columns = numerical_columns.index('income')

target_columns = numerical_columns[target_index_columns]

del numerical_columns[target_index_columns]

df[target_columns] = df[target_columns].map({"<=50K": 0, ">50K": 1})
df_test[target_columns] = df_test[target_columns].map({"<=50K": 0, ">50K": 1})

df[object_columns] = df[object_columns].astype('object')
df_test[object_columns] = df_test[object_columns].astype("object")

df[numerical_columns] = df[numerical_columns].astype(float)
df_test[numerical_columns] = df_test[numerical_columns].astype(float)

del df['fnlwgt']
del df_test["fnlwgt"]

In [45]:
df[object_columns] = df[object_columns].map(lambda x: x.lower().replace('-', '_'))
df_test[object_columns] = df_test[object_columns].map(
    lambda x: x.lower().replace("-", "_")
)

df.columns = df.columns.str.lower().str.replace("-", "_")
df_test.columns = df_test.columns.str.lower().str.replace("-", "_")

In [46]:
for index, text in enumerate(object_columns):
    object_columns[index] = text.replace("-", "_")

for index, text in enumerate(numerical_columns):
    numerical_columns[index] = text.replace("-", "_")

In [47]:
balanced_target_value_count = df['income'].value_counts().values

percent_balance_target_value = balanced_target_value_count[0] / balanced_target_value_count[1]

print(
    f"Percentage of balance of the target value in train data frame *(1/0) = {percent_balance_target_value}"
)

Percentage of balance of the target value in train data frame *(1/0) = 3.152659099604642


In [48]:
X_train, X_val, y_train, y_val = train_test_split(
    df, df["income"], test_size=0.2, random_state=RANDOM_STATE, stratify=df["income"]
)

df_shuffled = shuffle(df_test, random_state=RANDOM_STATE)

X_test = df_shuffled

In [49]:
y_train = X_train['income']
y_val = X_val['income']
y_test = X_test["income"]

In [50]:
del X_train['income']
del X_val["income"]
del X_test["income"]

In [51]:
X_train.reset_index(drop=True, inplace=True)
X_val.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)

y_train.reset_index(drop=True, inplace=True)
y_val.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)

C:\Users\TgSBot_xD\AppData\Local\Temp\ipykernel_13128\963717809.py:1: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  X_train.reset_index(drop=True, inplace=True)
C:\Users\TgSBot_xD\AppData\Local\Temp\ipykernel_13128\963717809.py:2: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  X_val.reset_index(drop=True, inplace=True)
C:\Users\TgSBot_xD\AppData\Local\Temp\ipykernel_13128\963717809.py:3: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  X_test.reset_index(drop=Tru

In [52]:
dicts_train = X_train.to_dict(orient='records')
dicts_validation = X_val.to_dict(orient="records")
dicts_test = X_test.to_dict(orient="records")

dv = DictVectorizer(sparse=False)

X_train_encode = dv.fit_transform(dicts_train)
X_val_encoded = dv.transform(dicts_validation)

features_name = dv.get_feature_names_out()
numerical_mask = np.isin(features_name, numerical_columns)

In [53]:
scaler = StandardScaler()

X_train_scaled = X_train_encode.copy()
X_val_scaled = X_val_encoded.copy()
X_train_scaled[:, numerical_mask] = scaler.fit_transform(X_train_encode[:, numerical_mask])
X_val_scaled[:, numerical_mask] = scaler.transform(X_val_encoded[:, numerical_mask])

print("Train:", X_train_scaled.shape, "Validation:", X_val_scaled.shape)

Train: (26048, 107) Validation: (6513, 107)


In [54]:
start_time_train_baseline_model = time.perf_counter()
baseline_model = float(y_train.mean())
stop_time_train_baseline_model = time.perf_counter()

time_train_baseline_model = stop_time_train_baseline_model - start_time_train_baseline_model

In [55]:
logistic_model_params = {"random_state": RANDOM_STATE}

logistic_model = LogisticRegression(**logistic_model_params)

start_time_train_logistic_model = time.perf_counter()
logistic_model.fit(X_train_scaled, y_train)
stop_time_train_logistic_model = time.perf_counter()

time_train_logistic_model = (
    stop_time_train_logistic_model - start_time_train_logistic_model
)


In [56]:
decision_tree_model_params = {"random_state": RANDOM_STATE}

decision_tree_model = DecisionTreeClassifier(**decision_tree_model_params)

start_time_train_decision_tree_model = time.perf_counter()
decision_tree_model.fit(X_train_encode, y_train)
stop_time_train_decision_tree_model = time.perf_counter()

time_train_decision_tree_model = (
    stop_time_train_decision_tree_model - start_time_train_decision_tree_model
)

In [57]:
random_forest_model_params = {"random_state": RANDOM_STATE}

random_forest_model = RandomForestClassifier(**random_forest_model_params)

start_time_train_random_forest_model = time.perf_counter()
random_forest_model.fit(X_train_encode, y_train)
stop_time_train_random_forest_model = time.perf_counter()

time_train_random_forest_model = (
    stop_time_train_random_forest_model - start_time_train_random_forest_model
)

In [58]:
xgboost_sklearn_model_params = {"random_state": RANDOM_STATE}

xgboost_sklearn_model = HistGradientBoostingClassifier(**xgboost_sklearn_model_params)

start_time_train_xgboost_sklearn_model = time.perf_counter()
xgboost_sklearn_model.fit(X_train_encode, y_train)
stop_time_train_xgboost_sklearn_model = time.perf_counter()

time_train_xgboost_sklearn_model = (
    stop_time_train_xgboost_sklearn_model - start_time_train_xgboost_sklearn_model
)

In [59]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train_encode, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val_encoded, label=y_val, feature_names=features)

xgboost_model_params = {
    "eta": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "binary:logistic",
    "nthread": 8,
    "seed": 1,
    "verbosity": 1,
}

xgboost_start_params = {'num_boost_round': 10}

xgboost_all_params = xgboost_start_params | xgboost_model_params

start_time_train_xgboost_model = time.perf_counter()
xgboost_model = xgb.train(xgboost_model_params, dtrain, **xgboost_start_params)
stop_time_train_xgboost_model = time.perf_counter()

time_train_xgboost_model = (
    stop_time_train_xgboost_model - start_time_train_xgboost_model
)

In [60]:
# baseline - roc_auc score

baseline_model_train_probabilities = np.full(len(y_train), baseline_model)
baseline_model_train_decisions = baseline_model_train_probabilities >= THRESHOLD

baseline_model_val_probabilities = np.full(len(y_val), baseline_model)
baseline_model_val_decisions = baseline_model_val_probabilities >= THRESHOLD

baseline_train_auc_score = 0.5
baseline_val_auc_score = 0.5

In [61]:
# logistic Regression - roc_auc score

logistic_model_train_roc_auc_score = roc_auc_score(y_train, logistic_model.predict_proba(X_train_scaled)[:, 1])
logistic_model_val_roc_auc_score = roc_auc_score(
    y_val, logistic_model.predict_proba(X_val_scaled)[:, 1]
)

In [62]:
# decision tree - roc_auc score

decision_tree_model_train_roc_auc_score = roc_auc_score(
    y_train, decision_tree_model.predict_proba(X_train_encode)[:, 1]
)
decision_tree_model_val_roc_auc_score = roc_auc_score(
    y_val, decision_tree_model.predict_proba(X_val_encoded)[:, 1]
)

In [63]:
# random_forest - roc_auc score

random_forest_model_train_roc_auc_score = roc_auc_score(
    y_train, random_forest_model.predict_proba(X_train_encode)[:, 1]
)
random_forest_model_val_roc_auc_score = roc_auc_score(
    y_val, random_forest_model.predict_proba(X_val_encoded)[:, 1]
)

In [64]:
# xgboost_sklearn - roc_auc score

xgboost_sklearn_model_train_roc_auc_score = roc_auc_score(
    y_train, xgboost_sklearn_model.predict_proba(X_train_encode)[:, 1]
)
xgboost_sklearn_model_val_roc_auc_score = roc_auc_score(
    y_val, xgboost_sklearn_model.predict_proba(X_val_encoded)[:, 1]
)

In [65]:
# xgboost - roc_auc score

xgboost_model_train_roc_auc_score = roc_auc_score(
    y_train, xgboost_model.predict(dtrain)
)
xgboost_model_val_roc_auc_score = roc_auc_score(
    y_val, xgboost_model.predict(dval)
)

In [66]:
# data models for a date frame

data_models = [
    {
        "method": "Baseline",
        "params": baseline_model,
        "train_roc_auc": baseline_train_auc_score,
        "val_roc_auc": baseline_val_auc_score,
        "time_train": time_train_baseline_model,
    },
    {
        "method": "LogisticRegression",
        "params": logistic_model_params,
        "train_roc_auc": logistic_model_train_roc_auc_score,
        "val_roc_auc": logistic_model_val_roc_auc_score,
        "time_train": time_train_logistic_model,
    },
    {
        "method": "Decision_Tree",
        "params": decision_tree_model_params,
        "train_roc_auc": decision_tree_model_train_roc_auc_score,
        "val_roc_auc": decision_tree_model_val_roc_auc_score,
        "time_train": time_train_decision_tree_model,
    },
    {
        "method": "Random_Forest",
        "params": random_forest_model_params,
        "train_roc_auc": random_forest_model_train_roc_auc_score,
        "val_roc_auc": random_forest_model_val_roc_auc_score,
        "time_train": time_train_random_forest_model,
    },
    {
        "method": "XGBoost_sklearn",
        "params": xgboost_sklearn_model_params,
        "train_roc_auc": xgboost_sklearn_model_train_roc_auc_score,
        "val_roc_auc": xgboost_sklearn_model_val_roc_auc_score,
        "time_train": time_train_xgboost_sklearn_model,
    },
    {
        "method": "XGBoost",
        "params": xgboost_all_params,
        "train_roc_auc": xgboost_model_train_roc_auc_score,
        "val_roc_auc": xgboost_model_val_roc_auc_score,
        "time_train": time_train_xgboost_model,
    },
]

df_statistic_models = pd.DataFrame(data_models)

In [67]:
FOLDER_OUTPUT.mkdir(exist_ok=True)
count_file_number = 0

for file_number in range(101):
    path = FOLDER_OUTPUT / f"try_{file_number}.csv"

    try:
        df_statistic_models.to_csv(
            path, mode="x", sep=";", encoding="utf-8", index=False
        )
    except FileExistsError:
        continue
    break


In [68]:
df_statistic_models

,method,params,train_roc_auc,val_roc_auc,time_train
0,Baseline,0.240825,0.500000,0.500000,0.000203
1,LogisticRegression,{'random_state': 30},0.908474,0.907579,0.177146
2,Decision_Tree,{'random_state': 30},0.998634,0.777047,0.225742
3,Random_Forest,{'random_state': 30},0.997679,0.890808,2.038616
4,XGBoost_sklearn,{'random_state': 30},0.944078,0.929562,0.688385
5,XGBoost,"{'num_boost_round': 10, 'eta': 0.3, 'max_depth...",0.924543,0.922007,0.046989
